In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from dotenv import load_dotenv
import joblib
from gene_thesaurus import GeneThesaurus


In [5]:
load_dotenv()

X_test_path = os.getenv("X_TEST_CSV_PATH")
y_test_path = os.getenv("Y_TEST_TSV_PATH")
X_test_feature_selected_path = os.getenv("X_TEST_FEATURE_SELECTED_CSV_PATH")
output_dir = os.getenv("OUTPUT_DIR")
random_seed = int(os.getenv("RANDOM_SEED"))
n_jobs = int(os.getenv("N_JOBS", -1))
rf_model_path = os.getenv("RF_MODEL_PATH")
selected_features_path = os.getenv("SELECTED_FEATURES_PATH")
pipeline_path = os.getenv("PIPELINE_PATH")
hyperparams_path = os.getenv("HYPERPARAMS_PATH")

# Print configuration
print(f"X_test_path: {X_test_path}")
print(f"y_test_path: {y_test_path}")
print(f"X_test_feature_selected_path: {X_test_feature_selected_path}")
print(f"output_dir: {output_dir}")
print(f"random_seed: {random_seed}")
print(f"n_jobs: {n_jobs}")
print(f"rf_model_path: {rf_model_path}")
print(f"selected_features_path: {selected_features_path}")
print(f"pipeline_path: {pipeline_path}")
print(f"hyperparams_path: {hyperparams_path}")

X_test_path: /home/mariya/Data/pallas/processed/dnam/dnam_test_X.csv
y_test_path: /home/mariya/Data/pallas/processed/dnam/dnam_test_y.tsv
X_test_feature_selected_path: /home/mariya/Data/pallas/processed/dnam/dnam_test_X_feature_selected.csv
output_dir: /home/mariya/Data/pallas/training/dnam
random_seed: 42
n_jobs: 10
rf_model_path: /home/mariya/Data/pallas/training/dnam/dnam_final_model.pkl
selected_features_path: /home/mariya/Data/pallas/training/dnam/dnam_selected_features.pkl
pipeline_path: /home/mariya/Data/pallas/training/dnam/dnam_pipeline.pkl
hyperparams_path: /home/mariya/Data/pallas/training/dnam/dnam_hyperparams.PKL


In [3]:
X_test = pd.read_csv(X_test_path, index_col=0)
y_test = pd.read_csv(y_test_path, sep="\t", index_col=0)
y_test

,subtype
id,
ALL_1154,"B-ALL, PAX5alt"
GSM5734757,"AML, KMT2A-r"
ALL_657,"B-ALL, high hyperdiploid"
GSM5734711,"AML, CBFB::MYH11"
ALL_1252,"B-ALL, high hyperdiploid"
...,...
ALL_1062,"B-ALL, MEF2D-r"
GSM5735128,"AML, DEK::NUP214"
ALL_944,"B-ALL, iAMP21"


In [4]:
# align the indices
X_test = X_test.loc[y_test.index]
X_test

,cg19895882,cg07124876,cg27267617,cg05301556,cg24430140,cg18475747,cg07067189,cg04324995,cg12868305,cg06103642,...,cg10983544,cg00028135,cg15658793,cg26470501,cg02207554,cg09634019,cg12128164,cg04561261,cg19109397,cg06967127
id,,,,,,,,,,,,,,,,,,,,,
ALL_1154,0.010735,0.851290,0.828931,0.898029,0.925337,0.581279,0.942538,0.059938,0.086359,0.221495,...,0.016729,0.942397,0.959595,0.446821,0.916756,0.932098,0.897496,0.914464,0.929876,0.866549
GSM5734757,0.288789,0.805903,0.843667,0.679263,0.018251,0.188391,0.899436,0.094367,0.111925,0.019504,...,0.031180,0.888483,0.782204,0.260031,0.616915,0.382772,0.735279,0.783694,0.890974,0.860531
ALL_657,0.202991,0.911846,0.746628,0.820948,0.282131,0.167795,0.932911,0.029689,0.431042,0.304304,...,0.187560,0.906532,0.893749,0.338623,0.800711,0.932255,0.860703,0.932114,0.951432,0.833265
GSM5734711,0.011166,0.768504,0.701235,0.769127,0.552687,0.036872,0.083034,0.050196,0.042073,0.017739,...,0.046925,0.947098,0.899113,0.498838,0.403128,0.945723,0.257232,0.840730,0.485305,0.918270
ALL_1252,0.028706,0.851273,0.655067,0.496961,0.371075,0.615505,0.922816,0.025897,0.044234,0.315649,...,0.194826,0.907820,0.876258,0.319254,0.440676,0.938698,0.558788,0.917314,0.872265,0.771112
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ALL_1062,0.010560,0.872726,0.436259,0.882052,0.921029,0.378541,0.681418,0.131637,0.031754,0.420776,...,0.524288,0.945552,0.967138,0.464272,0.935464,0.939868,0.937512,0.916365,0.931179,0.870516
GSM5735128,0.009121,0.653670,0.696865,0.767809,0.019094,0.236066,0.114238,0.030056,0.036461,0.037451,...,0.023202,0.885624,0.737962,0.445370,0.802695,0.532416,0.686856,0.801610,0.862949,0.879497
ALL_944,0.008619,0.915517,0.499182,0.890896,0.893347,0.483587,0.959914,0.016045,0.051727,0.021609,...,0.014151,0.938863,0.973131,0.436769,0.894165,0.944352,0.943227,0.923502,0.652001,0.564752


In [ ]:
# Load selected features
selected_features = joblib.load(selected_features_path)
X_test_selected = X_test[selected_features]


,cg07124876,cg07067189,cg01739725,cg11237495,cg10277811,cg27471989,cg22275872,cg03509771,cg08527226,cg07145988,...,cg05088898,cg09499856,cg01937808,cg12430029,cg20981451,cg10004780,cg11869770,cg19922137,cg01216388,cg17799946
id,,,,,,,,,,,,,,,,,,,,,
ALL_1154,0.851290,0.942538,0.950721,0.878240,0.910321,0.904852,0.896068,0.930047,0.824901,0.761829,...,0.106109,0.701635,0.936158,0.409506,0.812953,0.941896,0.596240,0.869049,0.278321,0.565912
GSM5734757,0.805903,0.899436,0.097319,0.731847,0.678789,0.398433,0.834762,0.896084,0.559936,0.847912,...,0.190383,0.013201,0.853394,0.788562,0.146752,0.076829,0.028647,0.011813,0.540085,0.022754
ALL_657,0.911846,0.932911,0.653037,0.885070,0.751230,0.883731,0.913067,0.909496,0.898097,0.769804,...,0.085862,0.537000,0.905804,0.479425,0.531536,0.674084,0.632141,0.648284,0.638407,0.618994
GSM5734711,0.768504,0.083034,0.400290,0.713807,0.775872,0.679314,0.128658,0.060866,0.298750,0.837054,...,0.130317,0.023572,0.925952,0.907965,0.303439,0.027711,0.221230,0.189593,0.462296,0.020728
ALL_1252,0.851273,0.922816,0.616707,0.875467,0.836878,0.860873,0.791443,0.916828,0.657288,0.677541,...,0.123059,0.541554,0.906431,0.174842,0.573884,0.732816,0.650132,0.697559,0.636121,0.529523
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ALL_1062,0.872726,0.681418,0.943831,0.874825,0.906218,0.904018,0.895146,0.910064,0.852886,0.952502,...,0.222639,0.707570,0.915788,0.638989,0.871357,0.951734,0.835975,0.891712,0.751155,0.840896
GSM5735128,0.653670,0.114238,0.840407,0.780117,0.652049,0.784059,0.658894,0.907624,0.531436,0.701468,...,0.148279,0.016416,0.854199,0.557266,0.099116,0.064059,0.078856,0.183516,0.558348,0.046381
ALL_944,0.915517,0.959914,0.941825,0.902001,0.883059,0.845273,0.912530,0.938887,0.869743,0.273895,...,0.096714,0.728846,0.945059,0.135900,0.882566,0.503166,0.611418,0.846180,0.739093,0.430099


In [7]:
# Save feature-selected test set for future reference
X_test_selected.to_csv(X_test_feature_selected_path, index=False)

In [ ]:
y_test = y_test.iloc[:, 0].values # Convert to 1D numpy array
y_test

array(['B-ALL, high hyperdiploid', 'B-ALL, BCR::ABL1',
       'B-ALL, ETV6::RUNX1', 'B-ALL, high hyperdiploid',
       'B-ALL, BCR::ABL1-like', 'B-ALL, ETV6::RUNX1',
       'B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1',
       'B-ALL, BCR::ABL1-like', 'B-ALL, BCR::ABL1-like',
       'B-ALL, high hyperdiploid', 'B-ALL, PAX5alt', 'B-ALL, ETV6::RUNX1',
       'B-ALL, BCR::ABL1', 'B-ALL, BCR::ABL1', 'B-ALL, PAX5alt',
       'B-ALL, DUX4-r', 'B-ALL, ETV6::RUNX1-like', 'B-ALL, TCF3::PBX1',
       'B-ALL, BCR::ABL1-like', 'B-ALL, KMT2A-r',
       'B-ALL, high hyperdiploid', 'B-ALL, TCF3::PBX1',
       'B-ALL, BCR::ABL1-like', 'B-ALL, KMT2A-r',
       'B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1-like',
       'B-ALL, TCF3::PBX1', 'B-ALL, BCR::ABL1-like', 'B-ALL, ZNF384-r',
       'B-ALL, ETV6::RUNX1', 'B-ALL, BCR::ABL1-like',
       'B-ALL, high hyperdiploid', 'B-ALL, MEF2D-r',
       'B-ALL, BCR::ABL1-like', 'B-ALL, BCR::ABL1-like',
       'B-ALL, TCF3::PBX1', 'B-ALL, ETV6::RUNX1',
    

In [42]:
N_ITER=30
N_REPEATS=1 # WE WILL VALIDATE FINAL PARAMS USING MORE REPEATS

# ----------------------
# Pipeline: Variance filter -> RF feature selection -> RF classifier
# ----------------------
pipeline = Pipeline([
    ("variance_filter", VarianceThreshold(threshold=VARIANCE_THRESHOLD)), # remove near-constant genes
    ("feature_selection", SelectPercentile(percentile=FEATURE_SELECTION_PERCENTILE)),  # keep only top 10% of features
    ("rf", RandomForestClassifier(random_seed, n_jobs=-n_jobs))
])

# ----------------------
# Hyperparameter search space
# ----------------------
param_grid = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [None, 20, 50],
    'rf__max_features': ['sqrt', 'log2'],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 5],
    'rf__class_weight': [None, 'balanced']
}

# ----------------------
# Cross-validation + randomized search
# ----------------------
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=N_ITER,
    cv=cv,
    scoring="f1_weighted",   # good for multiclass imbalance
    n_jobs=-n_jobs,
    random_state=random_seed,
    verbose=1
)

search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print("Best CV (F1-weighted):", search.best_score_)
best_pipeline = search.best_estimator_

## SAVE EVERYTHING
# Save best hyperparameters
joblib.dump(search.best_params_, os.path.join(output_dir, "best_hyperparameters.pkl"))

# Save features - USE BEST_PIPELINE INSTEAD OF PIPELINE
# First get features that passed the variance threshold
variance_mask = best_pipeline.named_steps['variance_filter'].get_support()
remaining_features = X_train.columns[variance_mask]

# Then get features selected by SelectFromModel
selected_mask = best_pipeline.named_steps['feature_selection'].get_support()
selected_features = remaining_features[selected_mask]

joblib.dump(selected_features, os.path.join(output_dir, "selected_features.pkl"))

# Save the whole pipeline
joblib.dump(best_pipeline, os.path.join(output_dir, "best_pipeline.pkl"))

Fitting 3 folds for each of 30 candidates, totalling 90 fits


Best hyperparameters: {'rf__n_estimators': 200, 'rf__min_samples_split': 10, 'rf__min_samples_leaf': 5, 'rf__max_features': 'sqrt', 'rf__max_depth': 50, 'rf__class_weight': 'balanced'}
Best CV (F1-weighted): 0.9125664037391447


['/home/mariya/Data/pallas/training/best_pipeline.pkl']

In [43]:
# Cross-validate again with more repeats and larger estimators
N_REPEATS=10

# Use just the selected features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# Read in "best_hyperparameters.pkl"
with open(os.path.join(output_dir, "best_hyperparameters.pkl"), "rb") as f:
    best_params = joblib.load(f)

best_params = {k.replace('rf__', ''): v for k, v in best_params.items()}
best_params['n_estimators'] = FINAL_N_ESTIMATORS
print("Final model parameters:", best_params)
final_rf = RandomForestClassifier(**best_params,
                            n_jobs=n_jobs,
                            random_state=random_seed)

# Cross-validate again with more repeats
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

cv_scores = cross_val_score(final_rf,
                           X_train_selected,
                           y_train,
                           cv=cv,
                           scoring="f1_weighted",
                           n_jobs=-n_jobs,
                           verbose=1)

print(f"Cross-validation F1-weighted: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Train the final model on all training data
final_rf.fit(X_train_selected, y_train)

# Save the final model
joblib.dump(final_rf, os.path.join(output_dir, "final_model.pkl"))

Final model parameters: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 50, 'class_weight': 'balanced'}


[Parallel(n_jobs=-10)]: Using backend LokyBackend with 5 concurrent workers.
[Parallel(n_jobs=-10)]: Done  30 out of  30 | elapsed:  3.8min finished


Cross-validation F1-weighted: 0.916 ± 0.009


['/home/mariya/Data/pallas/training/final_model.pkl']

In [44]:
# ----------------------
# Evaluate final model on test set
# ----------------------

# Use selected features for test set
y_pred_final = final_rf.predict(X_test_selected)

# Print size of selected features
print(f"Number of selected features: {len(selected_features)}")

# Print shape of X_test_selected
print(f"Shape of X_test_selected: {X_test_selected.shape}")

print("\nTest set classification report (final optimized model):")
print(classification_report(y_test, y_pred_final, digits=3))


Number of selected features: 3931
Shape of X_test_selected: (513, 3931)

Test set classification report (final optimized model):
                          precision    recall  f1-score   support

      AML, APL-PML::RARA      1.000     0.500     0.667         2
     AML, CBFA2T3::GLIS2      0.667     1.000     0.800         2
        AML, CBFB::MYH11      1.000     1.000     1.000         9
           AML, CEBPAalt      1.000     0.800     0.889         5
            AML, KMT2A-r      0.867     0.812     0.839        16
            AML, NPM1alt      1.000     0.625     0.769         8
            AML, NUP98-r      1.000     0.714     0.833         7
     AML, RUNX1::RUNX1T1      0.933     0.933     0.933        15
        B-ALL, BCR::ABL1      0.750     0.714     0.732        21
   B-ALL, BCR::ABL1-like      0.842     0.857     0.850        56
           B-ALL, DUX4-r      1.000     1.000     1.000        24
      B-ALL, ETV6::RUNX1      0.892     0.943     0.917        70
 B-ALL, ETV6

In [47]:

# ----------------------
# Extract top genes from final model
# ----------------------
# Get feature importances from final RF
importances = final_rf.feature_importances_

# Rank genes by importance
gene_importances = pd.DataFrame({
    "Gene": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

gt = GeneThesaurus()
translated = gt.translate_genes(gene_importances["Gene"].tolist(),
                               source="ensembl_id", target="symbol")
gene_importances["Gene"] = gene_importances["Gene"].map(translated)

print("\nTop 20 genes driving classification (final model):")
print(gene_importances.head(20))


Top 20 genes driving classification (final model):
         Gene  Importance
3268    NUTM1    0.005771
1889   MEGF10    0.005068
3830    APELA    0.003940
2203    AGAP1    0.003675
1830    MEIS1    0.003586
353   RUNX1T1    0.002926
706     KCNN1    0.002632
3315     PBX1    0.002585
3285   ANKS1B    0.002562
546     SALL4    0.002548
2161   TSPAN7    0.002514
2998  KBTBD11    0.002426
1036     CNN3    0.002212
3252   KCNJ12    0.002165
1413   ATP8A2    0.002147
2192     DGKI    0.002107
1436    MYH11    0.002062
348     HOXA9    0.002047
2247   DONSON    0.001997
1197    ATXN1    0.001973


In [48]:
# How many of the feature importances are zero?
num_zero_importances = np.sum(importances == 0)
print(f"\nNumber of features with zero importance: {num_zero_importances} out of {len(importances)}")

# Print lowest importance genes as well
print("\nBottom 20 genes by importance (final model):")
print(gene_importances.tail(20))


Number of features with zero importance: 4 out of 3931

Bottom 20 genes by importance (final model):
          Gene    Importance
2929     BRMS1  4.188505e-06
3807       MIF  4.140330e-06
70       APBA3  3.877452e-06
1373     BARX1  3.733394e-06
3340     FFAR4  3.618487e-06
3928      F8A1  3.532297e-06
460    LRRFIP2  2.825473e-06
1815   CREB3L4  2.743391e-06
3169    OR52K2  2.728419e-06
1932     FATE1  2.426159e-06
1172      CHD6  2.377005e-06
2846    DPAGT1  2.316390e-06
1774    SLC6A3  1.805014e-06
3471  KRTAP5-3  1.109546e-06
3907     ORAI1  1.773943e-20
3120     SOCS4  8.865781e-21
3665    XAGE1A  0.000000e+00
3366     QRFPR  0.000000e+00
3388      SBK2  0.000000e+00
937        IL4  0.000000e+00
